### 1. `init_chat_model` & API Keys
* **What it does:** Automatically detects and loads your API keys from your environment variables (like a `.env` file).
* **Why use it:** Keeps your code simple and prevents you from accidentally exposing private keys in your script.

### 2. The `temperature` Parameter
* **What it does:** Acts as a creativity dial (usually from `0.0` to `1.0`).
* **Low (0.0):** Strict, focused, and highly predictable. Best for coding, facts, and structured data.
* **High (0.7+):** Creative, random, and diverse. Best for brainstorming, writing, and casual conversation.

In [4]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
from langchain.chat_models import init_chat_model

# Switching to the newer, supported Gemini 3.1 lite model from your list
model = init_chat_model(model="gemini-3.1-flash-lite", model_provider="google_genai", temperature=0.7)
#model2=init_chat_model(model2="llama-2.1:lat")
print("success")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


success


# System and Human Messages
* Concept: Instead of just sending a raw string, LangChain allows you to categorize messages. A SystemMessage gives the AI its core     personality    or strict instructions, while a HumanMessage represents the user's input.

In [5]:
from langchain_core.messages import HumanMessage
from IPython.display import Markdown
# Use LangChain's native HumanMessage object
messages = [
    HumanMessage(content="tell me a fun fact")
]

# Invoke the model
response = model.invoke(messages)
response=response.content[0]['text']

# Safely print the content
display(Markdown(response))

A group of flamingos is called a **"flamboyance."**

In [6]:
from langchain.messages import SystemMessage
system_msg=SystemMessage('''You are a helpful assistant that responds to questions with the have a nice day.''')
human_message=HumanMessage("i am sangam how are you")
ans = model.invoke([system_msg,human_message])

In [7]:
ans=ans.content[0]['text']
display(Markdown(ans))

Hello Sangam! I am doing very well, thank you for asking. I hope you are having a wonderful time. Have a nice day!

# Prompt Templates
* Concept: Templates allow you to create reusable blueprints for your prompts. Instead of rewriting the entire paragraph every time, you define placeholders (like {context} and {question}) and dynamically inject variables into them later.

In [8]:
from langchain_core.prompts import PromptTemplate, prompt,ChatPromptTemplate
template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)
prompt=template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
 Language Models (LLMs). These models outperform their smaller
 counterparts and have become invaluable for developers who are creating
 applications with NLP capabilities. Developers can tap into these
 models through Hugging Face's `transformers` library, or by utilizing
 OpenAI and Cohere's offerings through the `openai` and `cohere`
 libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

In [9]:
completion=model.invoke(prompt)
print(completion)

content=[{'type': 'text', 'text': 'OpenAI and Cohere.', 'extras': {'signature': 'EjQKMgERTTIPTyJPw9zRzZ30UugNYE+kDYu3LZM0+RABS8Lc9kKK30Z5jfd06RZSNYiw7WjO'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f8455-7e53-7a21-b1e2-4256f3614f9b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 133, 'output_tokens': 6, 'total_tokens': 139, 'input_token_details': {'cache_read': 0}}


In [10]:
completion=completion.content[0]['text']
display(Markdown(completion))

OpenAI and Cohere.

In [11]:
template = ChatPromptTemplate.from_messages([
 ('system', '''Answer the question based on the context below. If the
 question cannot be answered using the information provided, answer
 with "I don\'t know".'''),
 ('human', 'Context: {context}'),
 ('human', 'Question: {question}'),
])
prompt=template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
 Language Models (LLMs). These models outperform their smaller
 counterparts and have become invaluable for developers who are creating
 applications with NLP capabilities. Developers can tap into these
 models through Hugging Face's `transformers` library, or by utilizing
 OpenAI and Cohere's offerings through the `openai` and `cohere`
 libraries, respectively.""",               
"question": "Which model providers offer LLMs?"})

In [12]:
model.invoke(prompt)


AIMessage(content=[{'type': 'text', 'text': 'OpenAI and Cohere.', 'extras': {'signature': 'EjQKMgERTTIPUuUNfgZDzf9A6qSpiKPqsQ65KBJSumGPTyakxBO+s4v7ifn/VECzCi/pHq3z'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8455-8d3d-7ac1-8b3c-00ce628fbea2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 6, 'total_tokens': 134, 'input_token_details': {'cache_read': 0}})

#  Structured Output (Pydantic)
* Concept: This forces the LLM to return data in a strict format (like a JSON object) instead of a conversational paragraph. (Note: I have updated this to use the modern Pydantic syntax, resolving the pydantic_v1 error you encountered earlier).

In [13]:
from langchain_core.tools import structured
from pydantic import BaseModel, Field
class Answerwithjustification(BaseModel):
    '''An answer to the user's question along with justification for the answer.'''
    answer: str
    '''The answer to the user's question'''
    justification: str
    '''Justification for the answer'''
llm=model.with_structured_output(Answerwithjustification)
llm.invoke("What weighs more, a pound of bricks or a pound of feathers")

Answerwithjustification(answer='They weigh the same.', justification='Both items weigh exactly one pound, meaning their mass is equal despite the difference in volume and density.')

# Output Parsers (Comma-Separated Lists)
* Concept: If you don't need a complex Pydantic model and just want a simple Python list back, output parsers format the raw text string into a usable Python data structure.

In [14]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser=CommaSeparatedListOutputParser()
items=parser.invoke("apple ,sangam, sharma")


In [15]:
items


['apple ', 'sangam', 'sharma']

In [16]:
completion=model.invoke('hi there')
completion.content[0]['text']

'Hello! How can I help you today?'

# Batching and Streaming
# Concept:

* Batch: Send multiple independent prompts to the AI at the exact same time (running in parallel), which is much faster than doing them in a for loop.

* Stream: Receive the AI's response chunk-by-chunk as it types, rather than waiting for the entire response to finish generating.

In [17]:
completions = model.batch(['Hi there!', 'Bye!'])
completions

[AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EjQKMgERTTIPHRivka+L3OS/NR3sXrPq14RkHammzA0uELHld+OmYsd0RARQci0Njz15l85/'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8455-b528-7951-81f2-fa60b50e7bba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 9, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}}),
 AIMessage(content=[{'type': 'text', 'text': 'Goodbye! Have a great day!', 'extras': {'signature': 'EjQKMgERTTIPDkOmjdFNQzZ8ryW1mt+nw3atWCVGfcfFILe3pqSJXAKvcy9R0+b+eVeP5Fg0'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8455-b52a-7c11-b2e0-48d59561a1ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadat

In [18]:
for token in model.stream('Bye!'):
    print(token)


content=[{'type': 'text', 'text': 'Bye! Have a great day!', 'index': 0}] additional_kwargs={} response_metadata={'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f8455-bb42-7b32-bfd8-0a59612a5cda' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 7, 'total_tokens': 10, 'input_token_details': {'cache_read': 0}} tool_call_chunks=[]
content=[{'type': 'text', 'text': '', 'extras': {'signature': 'EjQKMgERTTIPrnwvFJNZIV0AM725RcM0VMYBnwqRbyRm5VhV+/SYlfSoQktEyu7PRdzKq5wY'}, 'index': 0}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f8455-bb42-7b32-bfd8-0a59612a5cda' tool_calls=[] invalid_tool_calls=[] usage_metadata={'total_tokens': 0, 'input_token_details': {'cache_read': 0}, 'input_tokens': 0, 'output_tokens': 0} tool_call_chunks=[]
content=[] additional_kwargs={} response_metadata={} id='lc_run--019f8455-

# The @chain Decorator
* Concept: This turns a standard Python function into a "LangChain Runnable". It allows you to write custom logic inside a function, but still call it using standard LangChain methods like .invoke(), .batch(), or .stream().

In [19]:
from langchain_core.runnables import chain
template=ChatPromptTemplate.from_messages([
    ('system','you are a good assistent'),
    ('human','{question}')
])

@chain 
def chatbot(values):
    prompt=template.invoke(values)
    return model.invoke(prompt)

In [20]:
chatbot.invoke("what types of llm are here")

AIMessage(content=[{'type': 'text', 'text': 'Large Language Models (LLMs) can be categorized in several ways depending on their architecture, how they are trained, and what they are designed to do.\n\nHere is a breakdown of the main types of LLMs you will encounter today:\n\n### 1. By Architecture (The "Brain" Structure)\n*   **Decoder-Only Models:** These are the most common models today (e.g., GPT-4, Llama 3, Mistral). They are designed to predict the "next token" in a sequence. They are incredibly good at generative tasks like writing, coding, and chatting.\n*   **Encoder-Only Models:** These focus on "understanding" text rather than generating it. They are great for classification, sentiment analysis, and extracting information (e.g., BERT, RoBERTa).\n*   **Encoder-Decoder (Seq2Seq) Models:** These combine both. They encode an input and then decode it into a new output. They are excellent for translation or summarization tasks (e.g., T5, BART).\n\n### 2. By Accessibility (Who can u

### 7. LangChain Expression Language (LCEL) Using the Pipe `|`
* **What it does:** Connects components together in a single line like an assembly line. 
* **How it works:** The `|` operator takes the output of the component on the left (like a formatted prompt) and passes it directly as the input to the component on the right (like the model).
* **Why use it:** It replaces the need to write custom functions, keeping your code incredibly clean and easy to read.

In [21]:
template = ChatPromptTemplate.from_messages([
 ('system', 'You are a helpful assistant.'),
 ('human', '{question}'),
])
chatbot = template | model
chatbot.invoke({"question": "Which model providers offer LLMs?"})

AIMessage(content=[{'type': 'text', 'text': 'The landscape of Large Language Model (LLM) providers is vast and constantly evolving. These providers generally fall into three categories: **Model Builders** (who train their own models), **Model Hubs** (who host open-source and proprietary models), and **Cloud Platforms** (who provide infrastructure to run these models).\n\nHere is a breakdown of the primary providers:\n\n### 1. The "Big Three" (Proprietary Model Builders)\nThese companies build, train, and maintain their own state-of-the-art foundation models.\n*   **OpenAI:** The creator of the **GPT** series (GPT-4o, o1, GPT-4 Turbo). Accessible via ChatGPT Plus, the OpenAI API, and Microsoft Azure.\n*   **Google:** The creator of the **Gemini** family. Accessible via Gemini Advanced, Google AI Studio, Vertex AI, and the Gemini API.\n*   **Anthropic:** The creator of the **Claude** series (Claude 3.5 Sonnet, Claude 3 Opus/Haiku). Accessible via the Claude web interface, the Anthropic A